# Chapter 7: Relational Database Design and Normalization


## Core Question

Which facts are stored repeatedly, can a decomposition reconstruct the original relation
without false rows, and does every important determinant identify a complete row?

Use this guide with the executable SQL lab cell. Predict before executing, and do not infer a
business rule from one convenient sample instance.


## Scope and Connection

Chapter 6 mapped requirements to relations. This chapter is useful when an existing table,
spreadsheet, or mixed design still combines several kinds of facts.

The classroom core is anomalies, functional dependencies, attribute closure, binary
lossless decomposition, spurious tuples, and introductory 3NF/BCNF decisions. Canonical
covers and complete decomposition algorithms are extensions.


## Teaching Summary

| Topic | Worked example and practice | Evidence to retain |
|---|---|---|
| Anomalies and dependencies | Flattened course facts | Business rule and counterexample |
| Attribute closure | Candidate-key calculation | Closure steps and minimality check |
| Lossless decomposition | Reconstructing normalized relations | Both difference checks |
| 3NF and BCNF | Small dependency sets | Determinant and key reasoning |


## Prerequisites

- Relations, tuples, candidate keys, and foreign keys.
- Projection and natural join.
- Basic schema and data-modification SQL.
- Chapter 6 entity, relationship, and mapping concepts.


## Learning Objectives

After completing this chapter, you should be able to:

1. Identify update, insertion, and deletion anomalies.
2. Write a functional dependency from a stable business rule.
3. Reject an unsupported dependency with a counterexample.
4. Calculate a small attribute closure and test key minimality.
5. Test a binary decomposition for losslessness and identify spurious tuples.
6. Apply introductory BCNF and 3NF criteria.
7. Explain a basic tradeoff involving BCNF and dependency preservation.


## 1. Mixed Facts and Anomalies

Consider:

```text
course_enrollment_record(
  student_id, student_name, dept_code, dept_name,
  course_id, course_title, credits, grade
)
```

Business rules imply:

```text
student_id -> student_name, dept_code
dept_code  -> dept_name
course_id  -> course_title, credits
(student_id, course_id) -> grade
```

This relation stores Student, Department, Course, and Enrollment facts together.

- **Update anomaly:** changing one repeated department name can create inconsistent copies.
- **Insertion anomaly:** a course with no enrollment cannot be stored naturally.
- **Deletion anomaly:** deleting the last enrollment in a course can remove the only copy
  of that course's title and credits.

The problem is not the number of columns. It is the dependency structure and the resulting
operations.

### Practice

For `employee_project(employee_id, employee_name, project_id, project_name, hours)`,
identify Employee, Project, and assignment facts and one anomaly of each applicable type.


## 2. Functional Dependencies

For schema R, `alpha -> beta` means that every legal pair of tuples that agrees on all
attributes in alpha must also agree on beta. Alpha is the determinant.

A dependency such as `(student_id, course_id) -> student_id` is trivial because the right
side is included in the left. If `K -> R`, K is a superkey; if K is minimal, it is a
candidate key.

### Rule and Counterexample

The rule "each student identifier represents one student" supports:

```text
student_id -> student_name, dept_code
```

It does not support `student_id -> course_id`. The legal rows `(S101, DB201)` and
`(S101, FT210)` form a counterexample.

Practice: decide whether `course_title -> course_id` is supported. Either state a rule
that guarantees unique titles or provide two legal courses with one title and different
identifiers.


## 3. Attribute Closure and Candidate Keys

The closure `alpha+` contains every attribute functionally determined by alpha. Start with
alpha, repeatedly apply dependencies whose left sides are already present, and stop when
no new attribute can be added.

For `{student_id, course_id}`:

1. Begin with both identifiers.
2. Add student name and department code from `student_id`.
3. Add department name from department code.
4. Add course title and credits from `course_id`.
5. Add grade from the identifier pair.

The closure contains every attribute, so the pair is a superkey. Removing either
identifier prevents determination of the other entity's facts and grade, so the pair is
minimal and is therefore a candidate key.

### Practice

Given:

```text
employee_id -> employee_name
project_id -> project_name
(employee_id, project_id) -> hours
```

Calculate `{employee_id, project_id}+` and test both single-attribute removals.


## 4. Lossless Decomposition

A decomposition is lossless if projecting a legal relation into the new schemas and
joining the projections reconstructs exactly the original relation.

For a binary decomposition R into R1 and R2 under functional dependencies, the step is
lossless when the common attributes determine all attributes of R1 or all attributes of
R2:

```text
(R1 intersection R2) -> R1
or
(R1 intersection R2) -> R2
```

Having a shared column is not enough; the shared attributes must identify one side.

### Lossless Example

Split Department from the flattened relation:

```text
Department(dept_code, dept_name)
Remaining(student_id, student_name, dept_code,
          course_id, course_title, credits, grade)
```

The common attribute `dept_code` determines Department, so this step is lossless. Further
steps produce Department, Student, Course, and Enrollment relations. The lab uses
bidirectional `EXCEPT` checks for the supplied instance.

### Lossy Example

Original rows:

```text
(E1, Kim, Taipei, 60000)
(E2, Kim, Tainan, 62000)
```

Split into `EmployeeIdentity(employee_id, name)` and
`EmployeeDetails(name, city, salary)`. Name is not a key. Joining the projections produces
four rows, including two false employee-city combinations.

### Practice

Run the lossless and lossy lab sections. Retain both difference counts for the lossless
case and identify the two spurious Kim rows in the lossy case.


## 5. Boyce-Codd Normal Form

R is in BCNF when every nontrivial dependency `alpha -> beta` has a determinant alpha
that is a superkey.

In the flattened relation, `dept_code -> dept_name` is nontrivial, but department code
does not determine Student, Course, and Enrollment facts. This is a BCNF violation.

After decomposition:

- Department has key determinant `dept_code`.
- Student has key determinant `student_id`.
- Course has key determinant `course_id`.
- Enrollment has key determinant `(student_id, course_id)`.

Under the listed rules, each relation satisfies BCNF. A new business rule requires a new
check.


## 6. Third Normal Form

For every dependency `alpha -> beta`, 3NF allows the dependency when it is trivial, alpha
is a superkey, or each attribute in `beta - alpha` is prime. A prime attribute occurs in
at least one candidate key. Every BCNF relation is in 3NF, but not every 3NF relation is
in BCNF.

### 3NF but Not BCNF Example

```text
TeachingAssignment(student_id, course_id, instructor_id)

(student_id, course_id) -> instructor_id
instructor_id -> course_id
```

If every instructor teaches only one course, candidate keys are
`(student_id, course_id)` and `(student_id, instructor_id)`. All three attributes are
prime. `instructor_id -> course_id` violates BCNF because instructor ID is not a superkey,
but it satisfies the prime-attribute allowance in 3NF.

If instructors may teach several courses, the second dependency no longer holds and the
normal-form analysis changes. Normal form is a property of a schema together with its
valid dependencies.


## 7. Dependency Preservation as a Tradeoff

A dependency-preserving decomposition allows each original dependency to be checked in
individual decomposed relations without joining them.

Splitting TeachingAssignment into
`InstructorCourse(instructor_id, course_id)` and
`StudentInstructor(student_id, instructor_id)` can be lossless and BCNF under the stated
rule. However, the original dependency
`(student_id, course_id) -> instructor_id` is not contained in one relation and cannot be
checked with one local key constraint.

This example shows why a design discussion should include redundancy, losslessness,
dependency enforcement, and implementation cost rather than stopping after the label
BCNF.


## Common Errors

1. Guessing a dependency from a small sample without a business rule.
2. Treating current uniqueness as proof of a candidate key.
3. Listing decomposed tables without anomalies, dependencies, or a lossless argument.
4. Assuming that any shared attribute makes a join lossless.
5. Applying a simplified 3NF slogan without finding keys and prime attributes.
6. Assuming that BCNF always preserves every dependency.
7. Treating one successful sample join as proof for every legal instance.


## Classroom and Individual Evidence

Compare a 3NF single relation with a BCNF decomposition using the same four criteria:
repetition, losslessness, local dependency checking, and implementation cost.

Submit the original relation and rules, dependencies and key closure, one anomaly, the
decomposition and lossless reason, normal-form decisions, and one revision made after
feedback.


## Chapter Summary

Normalization converts business rules into functional dependencies and uses closure,
lossless decomposition, and normal-form criteria to evaluate a schema. A good answer
preserves both data meaning and enforceable constraints. Chapter 14 moves to physical
design and query-plan evidence after the logical design is sound.


## After-Class Continuation

Repeat the sequence "rules, dependencies, anomalies, decomposition, lossless check" on a
real table. Armstrong's axioms, canonical covers, and complete decomposition algorithms
are extensions rather than Exam 2 operations.


## Build and Inspect the Chapter Database

The executable cells use SQLite through Python's standard `sqlite3` module. Follow the
cells in order:

1. Open a database connection and enable foreign-key enforcement.
2. Execute the chapter's `CREATE TABLE` statements before inserting rows.
3. Load the synthetic example data.
4. Inspect the resulting tables, columns, primary keys, and foreign keys.
5. Check referential integrity before running the chapter queries.
6. Predict each query result, execute it, and explain any difference.

`DATABASE_NAME` is initially `:memory:`, so closing the notebook removes the database.
Change it to a filename such as `chapter_database.db` when you want SQLite to create a
persistent database in the notebook's working directory. Do not switch to a persistent
file until the in-memory version runs successfully from top to bottom.


In [1]:
import sqlite3

print(f"Python {__import__('sys').version.split()[0]}; SQLite {sqlite3.sqlite_version}")
DATABASE_NAME = ":memory:"  # Change to "chapter_database.db" to keep a database file.
connection = sqlite3.connect(DATABASE_NAME, isolation_level=None)
connection.execute("PRAGMA foreign_keys = ON")


def run_sql_script(connection, script, max_rows=20):
    """Execute a SQLite script and display result-producing statements."""
    buffer = ""
    for raw_line in script.splitlines():
        stripped = raw_line.strip()
        if stripped.startswith(".print"):
            message = stripped[len(".print"):].strip().strip("\"'")
            print(f"\n{message}")
            continue
        buffer += raw_line + "\n"
        if not sqlite3.complete_statement(buffer):
            continue
        statement = buffer.strip()
        buffer = ""
        if not statement:
            continue
        cursor = connection.execute(statement)
        if cursor.description:
            columns = [column[0] for column in cursor.description]
            rows = cursor.fetchmany(max_rows + 1)
            print(" | ".join(columns))
            for row in rows[:max_rows]:
                print(" | ".join("NULL" if value is None else str(value) for value in row))
            if len(rows) > max_rows:
                print(f"... additional rows omitted after {max_rows}")
    remaining = "\n".join(
        line for line in buffer.splitlines() if not line.strip().startswith("--")
    ).strip()
    if remaining:
        raise ValueError("The embedded SQL ends with an incomplete statement.")


def inspect_database(connection):
    """Display tables, columns, primary keys, foreign keys, and integrity status."""
    tables = [
        row[0]
        for row in connection.execute(
            "SELECT name FROM sqlite_master WHERE type='table' AND name NOT LIKE 'sqlite_%' ORDER BY name"
        )
    ]
    print("Tables:", ", ".join(tables) if tables else "none")
    for table in tables:
        columns = connection.execute(f'PRAGMA table_info("{table}")').fetchall()
        primary_key = [row[1] for row in sorted(columns, key=lambda row: row[5]) if row[5]]
        print(f"\n{table}")
        print("  columns:", ", ".join(f"{row[1]} {row[2]}" for row in columns))
        print("  primary key:", ", ".join(primary_key) if primary_key else "none")
        for index_row in connection.execute(f'PRAGMA index_list("{table}")').fetchall():
            if index_row[2] and index_row[3] == "u":
                unique_columns = [
                    row[2]
                    for row in connection.execute(
                        f'PRAGMA index_info("{index_row[1]}")'
                    ).fetchall()
                ]
                print("  unique constraint:", ", ".join(unique_columns))
        foreign_keys = connection.execute(f'PRAGMA foreign_key_list("{table}")').fetchall()
        for foreign_key in foreign_keys:
            print(f"  foreign key: {foreign_key[3]} -> {foreign_key[2]}.{foreign_key[4]}")
    violations = connection.execute("PRAGMA foreign_key_check").fetchall()
    print("\nForeign-key check:", "PASS" if not violations else violations)


Python 3.12.13; SQLite 3.53.1


### Normalization lab


In [2]:
SQL_1 = """PRAGMA foreign_keys = ON;

DROP TABLE IF EXISTS ch07_enrollment;
DROP TABLE IF EXISTS ch07_course;
DROP TABLE IF EXISTS ch07_student;
DROP TABLE IF EXISTS ch07_department;
DROP TABLE IF EXISTS ch07_course_enrollment_record;
DROP TABLE IF EXISTS ch07_employee_details;
DROP TABLE IF EXISTS ch07_employee_identity;

-- Part A: A flattened relation that stores several kinds of facts.
CREATE TABLE ch07_course_enrollment_record (
    student_id   TEXT NOT NULL,
    student_name TEXT NOT NULL,
    dept_code    TEXT NOT NULL,
    dept_name    TEXT NOT NULL,
    course_id    TEXT NOT NULL,
    course_title TEXT NOT NULL,
    credits      INTEGER NOT NULL,
    grade        TEXT,
    PRIMARY KEY (student_id, course_id)
);

INSERT INTO ch07_course_enrollment_record VALUES
    ('S101', 'An Chen',  'IM',  'Information Management', 'DB201', 'Database Management', 3, 'A'),
    ('S101', 'An Chen',  'IM',  'Information Management', 'FT210', 'Financial Technology', 3, 'B+'),
    ('S102', 'Bea Lin',  'FIN', 'Finance',                'FT210', 'Financial Technology', 3, 'A-'),
    ('S103', 'Kai Wu',   'IM',  'Information Management', 'DB201', 'Database Management', 3, 'B');

SELECT dept_code, dept_name, COUNT(*) AS repeated_rows
FROM ch07_course_enrollment_record
GROUP BY dept_code, dept_name
ORDER BY dept_code;

-- Demonstrate an update anomaly without retaining invalid data.
SAVEPOINT inconsistent_department_name;
UPDATE ch07_course_enrollment_record
SET dept_name = 'Information Systems'
WHERE student_id = 'S101' AND course_id = 'DB201';

SELECT dept_code, COUNT(DISTINCT dept_name) AS distinct_names
FROM ch07_course_enrollment_record
GROUP BY dept_code
ORDER BY dept_code;

ROLLBACK TO inconsistent_department_name;
RELEASE inconsistent_department_name;

-- Part B: Store each kind of fact once.
CREATE TABLE ch07_department (
    dept_code TEXT PRIMARY KEY,
    dept_name TEXT NOT NULL UNIQUE
);

CREATE TABLE ch07_student (
    student_id   TEXT PRIMARY KEY,
    student_name TEXT NOT NULL,
    dept_code    TEXT NOT NULL,
    FOREIGN KEY (dept_code) REFERENCES ch07_department(dept_code)
);

CREATE TABLE ch07_course (
    course_id    TEXT PRIMARY KEY,
    course_title TEXT NOT NULL,
    credits      INTEGER NOT NULL CHECK (credits BETWEEN 1 AND 6)
);

CREATE TABLE ch07_enrollment (
    student_id TEXT NOT NULL,
    course_id  TEXT NOT NULL,
    grade      TEXT CHECK (grade IS NULL OR grade IN
        ('A', 'A-', 'B+', 'B', 'B-', 'C+', 'C', 'C-', 'D', 'F')),
    PRIMARY KEY (student_id, course_id),
    FOREIGN KEY (student_id) REFERENCES ch07_student(student_id),
    FOREIGN KEY (course_id) REFERENCES ch07_course(course_id)
);

INSERT INTO ch07_department VALUES
    ('IM', 'Information Management'),
    ('FIN', 'Finance');

INSERT INTO ch07_student VALUES
    ('S101', 'An Chen', 'IM'),
    ('S102', 'Bea Lin', 'FIN'),
    ('S103', 'Kai Wu', 'IM');

INSERT INTO ch07_course VALUES
    ('DB201', 'Database Management', 3),
    ('FT210', 'Financial Technology', 3),
    ('AI301', 'Artificial Intelligence', 3);

INSERT INTO ch07_enrollment VALUES
    ('S101', 'DB201', 'A'),
    ('S101', 'FT210', 'B+'),
    ('S102', 'FT210', 'A-'),
    ('S103', 'DB201', 'B');

-- AI301 can exist before any student enrolls.
SELECT c.course_id, c.course_title, COUNT(e.student_id) AS enrollment_count
FROM ch07_course AS c
LEFT JOIN ch07_enrollment AS e ON e.course_id = c.course_id
GROUP BY c.course_id, c.course_title
ORDER BY c.course_id;

-- Part C: Reconstruct the original four rows and compare in both directions.
WITH reconstructed AS (
    SELECT s.student_id, s.student_name, d.dept_code, d.dept_name,
           c.course_id, c.course_title, c.credits, e.grade
    FROM ch07_enrollment AS e
    JOIN ch07_student AS s ON s.student_id = e.student_id
    JOIN ch07_department AS d ON d.dept_code = s.dept_code
    JOIN ch07_course AS c ON c.course_id = e.course_id
)
SELECT COUNT(*) AS original_minus_reconstructed
FROM (
    SELECT * FROM ch07_course_enrollment_record
    EXCEPT
    SELECT * FROM reconstructed
);

WITH reconstructed AS (
    SELECT s.student_id, s.student_name, d.dept_code, d.dept_name,
           c.course_id, c.course_title, c.credits, e.grade
    FROM ch07_enrollment AS e
    JOIN ch07_student AS s ON s.student_id = e.student_id
    JOIN ch07_department AS d ON d.dept_code = s.dept_code
    JOIN ch07_course AS c ON c.course_id = e.course_id
)
SELECT COUNT(*) AS reconstructed_minus_original
FROM (
    SELECT * FROM reconstructed
    EXCEPT
    SELECT * FROM ch07_course_enrollment_record
);

-- Part D: A deliberately lossy decomposition using a non-key common attribute.
CREATE TABLE ch07_employee_identity (
    employee_id TEXT PRIMARY KEY,
    name        TEXT NOT NULL
);

CREATE TABLE ch07_employee_details (
    name   TEXT NOT NULL,
    city   TEXT NOT NULL,
    salary INTEGER NOT NULL,
    PRIMARY KEY (name, city)
);

INSERT INTO ch07_employee_identity VALUES
    ('E1', 'Kim'),
    ('E2', 'Kim');

INSERT INTO ch07_employee_details VALUES
    ('Kim', 'Taipei', 60000),
    ('Kim', 'Tainan', 62000);

SELECT i.employee_id, i.name, d.city, d.salary
FROM ch07_employee_identity AS i
JOIN ch07_employee_details AS d USING (name)
ORDER BY i.employee_id, d.city;
"""


In [3]:
run_sql_script(connection, SQL_1)


dept_code | dept_name | repeated_rows
FIN | Finance | 1
IM | Information Management | 3
dept_code | distinct_names
FIN | 1
IM | 2
course_id | course_title | enrollment_count
AI301 | Artificial Intelligence | 0
DB201 | Database Management | 2
FT210 | Financial Technology | 2
original_minus_reconstructed
0
reconstructed_minus_original
0
employee_id | name | city | salary
E1 | Kim | Tainan | 62000
E1 | Kim | Taipei | 60000
E2 | Kim | Tainan | 62000
E2 | Kim | Taipei | 60000


### Inspect the Database You Created

Read the output as a schema check: confirm the table names, column types, primary-key order, foreign-key direction, and integrity result.


In [4]:
inspect_database(connection)


Tables: ch07_course, ch07_course_enrollment_record, ch07_department, ch07_employee_details, ch07_employee_identity, ch07_enrollment, ch07_student

ch07_course
  columns: course_id TEXT, course_title TEXT, credits INTEGER
  primary key: course_id

ch07_course_enrollment_record
  columns: student_id TEXT, student_name TEXT, dept_code TEXT, dept_name TEXT, course_id TEXT, course_title TEXT, credits INTEGER, grade TEXT
  primary key: student_id, course_id

ch07_department
  columns: dept_code TEXT, dept_name TEXT
  primary key: dept_code
  unique constraint: dept_name

ch07_employee_details
  columns: name TEXT, city TEXT, salary INTEGER
  primary key: name, city

ch07_employee_identity
  columns: employee_id TEXT, name TEXT
  primary key: employee_id

ch07_enrollment
  columns: student_id TEXT, course_id TEXT, grade TEXT
  primary key: student_id, course_id
  foreign key: course_id -> ch07_course.course_id
  foreign key: student_id -> ch07_student.student_id

ch07_student
  columns: stude

### Reproducibility Check


In [5]:
assert connection.execute("SELECT COUNT(*) FROM ch07_course_enrollment_record").fetchone()[0] == 4
lossy_count = connection.execute("SELECT COUNT(*) FROM ch07_employee_identity JOIN ch07_employee_details USING (name)").fetchone()[0]
assert lossy_count == 4
assert connection.execute("PRAGMA foreign_key_check").fetchall() == []
print("Notebook checks passed.")


Notebook checks passed.


In [6]:
connection.close()
print("In-memory database closed.")


In-memory database closed.
